# <font color="#418FDE" size="6.5" uppercase>**Pandas & Arrays**</font>

>Last update: 20260315.
    
By the end of this Lecture, you will be able to:
- Load and visualize basic civil engineering data using Pandas, NumPy, matplotlib, and seaborn. 
- Perform simple numerical operations such as converting time-series from time domain to frequency domain or extracting time-series features. 
- Use NumPy arrays to compute vectorized statistics and transformations on engineering measurements. 


## **1. NumPy Data Visualization**

### **1.1. NumPy Array Operations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_B/image_01_01.jpg?v=1773550148" width="250">



>* Clean, align, transform measurements before plotting.
>* Reshape, slice, mask, fix units early.

>* Vectorized transforms quickly standardize, detrend, and normalize.
>* Combine and broadcast signals for clearer plots.

>* Flag bad sensor data; compute plot-ready summaries.
>* Align arrays for consistent, repeatable visualizations.



In [ ]:
#@title Python Code - NumPy Array Operations

# Learn core NumPy operations for sensor arrays.
# Prepare clean arrays before plotting engineering signals.
# Visualize transformed channels using a simple heatmap.

# pip install numpy pandas matplotlib seaborn.

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(7)
fs_hz, seconds, channels = 50, 10, 4

# Create synthetic accelerometer data with drift.
t = np.arange(fs_hz * seconds) / fs_hz
base = 0.15 * np.sin(2 * np.pi * 2 * t)

# Stack channels with different amplitudes and noise.
noise = 0.05 * np.random.randn(t.size, channels)
scale = np.array([1.0, 0.7, 1.3, 0.9])

signals = base[:, None] * scale[None, :] + noise
signals = signals + (0.02 * t)[:, None]

# Inject missing values and a spike artifact.
signals[120, 2] = np.nan
signals[260, 1] = 2.5

# Replace NaNs using per-channel mean values.
col_mean = np.nanmean(signals, axis=0)
mask_nan = np.isnan(signals)

signals[mask_nan] = np.take(col_mean, np.where(mask_nan)[1])
valid = np.abs(signals) < 1.5

# Mask outliers and compute per-channel baseline.
masked = np.where(valid, signals, np.nan)
baseline = np.nanmean(masked, axis=0)

# Detrend and normalize each channel for comparison.
detrended = signals - baseline[None, :]
std = np.std(detrended, axis=0)

std = np.where(std > 1e-9, std, 1.0)
normalized = detrended / std[None, :]

# Slice a time window around an event.
start, stop = int(2 * fs_hz), int(6 * fs_hz)
window = normalized[start:stop, :]

# Compute vectorized features for quick reporting.
rms = np.sqrt(np.mean(window ** 2, axis=0))
peak = np.max(np.abs(window), axis=0)

print("Window shape:", window.shape)
print("RMS per channel:", np.round(rms, 3))
print("Peak per channel:", np.round(peak, 3))

# Plot a heatmap showing normalized amplitudes over time.
plt.figure(figsize=(8, 3.6))
sns.heatmap(window.T, cmap="coolwarm", center=0, cbar=True)

plt.xlabel("Sample index in selected window")
plt.ylabel("Channel index")
plt.title("NumPy array operations: clean, slice, normalize")

plt.tight_layout()
plt.show()



### **1.2. Array Feature Extraction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_B/image_01_02.jpg?v=1773550174" width="250">



>* Summarize raw sensor signals into key features.
>* Use NumPy stats to spot trends, glitches.

>* Choose plot questions; compute matching array features.
>* Use robust, rolling features for comparisons.

>* Standardized features compare different sampling and lengths
>* Feature plots reveal behavior despite noise



In [ ]:
#@title Python Code - Array Feature Extraction

# Learn array features for civil engineering signals.
# Use NumPy and Pandas for quick summaries.
# Visualize extracted features with one clear plot.

# !pip -q install pandas numpy matplotlib seaborn

# Import core tools for arrays and plots.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set a seed for repeatable synthetic measurements.
np.random.seed(7)

# Create a short vibration-like acceleration time series.
fs_hz = 50
seconds = 20
n = int(fs_hz * seconds)
t = np.arange(n) / fs_hz

# Combine a sinusoid, noise, and one short event.
f0_hz = 3.0
signal = 0.25 * np.sin(2 * np.pi * f0_hz * t)
signal = signal + 0.05 * np.random.randn(n)

# Add a brief spike event like a passing truck.
event_start = int(8.0 * fs_hz)
event_end = int(8.4 * fs_hz)
signal[event_start:event_end] = signal[event_start:event_end] + 0.6

# Insert missing points to mimic telemetry dropouts.
missing_idx = np.random.choice(n, size=12, replace=False)
signal[missing_idx] = np.nan

# Convert to a Pandas series for easy cleaning.
acc = pd.Series(signal, name="acc_mps2")

# Fill missing values using simple linear interpolation.
acc_clean = acc.interpolate(limit_direction="both")

# Convert cleaned series into a NumPy array.
x = acc_clean.to_numpy()

# Validate array size before computing features.
assert x.ndim == 1 and x.size > 10

# Compute basic magnitude and spread features.
mean_val = float(np.mean(x))
std_val = float(np.std(x, ddof=1))
rms_val = float(np.sqrt(np.mean(x ** 2)))

# Compute robust and extreme features for quality checks.
min_val = float(np.min(x))
max_val = float(np.max(x))
p95_val = float(np.percentile(x, 95))

# Compute a simple shape feature using zero crossings.
zc = int(np.sum(np.diff(np.signbit(x)) != 0))

# Compute exceedance count for a serviceability threshold.
threshold = 0.35
exceed = int(np.sum(np.abs(x) > threshold))

# Summarize features into a small, plot-ready table.
features = pd.DataFrame(
    {
        "feature": [
            "mean","std","rms","min","max","p95","zero_crossings","exceed_count",
        ],
        "value": [
            mean_val,std_val,rms_val,min_val,max_val,p95_val,float(zc),float(exceed),
        ],
    }
)

# Print a compact feature summary for quick inspection.
print("Extracted features from one accelerometer record:")
print(features.head(8).to_string(index=False))

# Plot features to compare magnitudes at a glance.
plt.figure(figsize=(9, 4))
sns.barplot(data=features, x="feature", y="value", color="steelblue")
plt.title("Array Feature Extraction: Compact Signal Descriptors")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


### **1.3. Civil Data Plotting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_B/image_01_03.jpg?v=1773550211" width="250">



>* Plot arrays with axes, units, and context.
>* Spot issues, choose line/scatter, guide preprocessing.

>* Overlay signals; use clear colors and axes.
>* Use seaborn distributions; tidy data for faceting.

>* Link plots to space and alert thresholds.
>* Use clear units, scales, uncertainty for decisions.



In [ ]:
#@title Python Code - Civil Data Plotting

# Learn plotting for civil sensor arrays today.
# Use NumPy, Pandas, and Seaborn together.
# Create one diagnostic plot with units.

# !pip -q install seaborn pandas matplotlib numpy.

# Import core libraries for arrays and plotting.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set a deterministic seed for reproducible results.
rng = np.random.default_rng(7)

# Create timestamps for a short monitoring window.
fs_hz = 50
seconds = 20
n = int(fs_hz * seconds)

# Build a time axis in seconds.
t = np.arange(n) / fs_hz

# Simulate bridge acceleration with noise and drift.
base = 0.03 * np.sin(2 * np.pi * 2.0 * t)
noise = 0.01 * rng.standard_normal(n)
drift = 0.0005 * t

# Add a few impact spikes as outliers.
acc = base + noise + drift
spike_idx = rng.choice(n, size=6, replace=False)
acc[spike_idx] = acc[spike_idx] + rng.normal(0.12, 0.03, 6)

# Validate array sizes before building a dataframe.
assert t.size == acc.size and t.size > 10

# Organize arrays into a tidy Pandas dataframe.
df = pd.DataFrame({"time_s": t, "acc_g": acc})

# Compute simple features for quick diagnostics.
mean_g = float(np.mean(acc))
std_g = float(np.std(acc))
max_g = float(np.max(np.abs(acc)))

# Print a compact summary without dumping arrays.
print(f"Samples: {n}, Sampling rate: {fs_hz} Hz")
print(f"Mean: {mean_g:.4f} g, Std: {std_g:.4f} g")
print(f"Max absolute: {max_g:.4f} g")

# Choose an alert threshold for visual reference.
alert_g = 0.08

# Use seaborn styling for readable engineering plots.
sns.set_theme(style="whitegrid")

# Create one figure with a clear time axis.
plt.figure(figsize=(10, 4))

# Plot the time series as a continuous line.
sns.lineplot(data=df, x="time_s", y="acc_g", linewidth=1.2)

# Overlay spikes as points to highlight outliers.
plt.scatter(df.loc[spike_idx, "time_s"], df.loc[spike_idx, "acc_g"], s=35)

# Add threshold lines to support decisions.
plt.axhline(alert_g, color="red", linestyle="--")
plt.axhline(-alert_g, color="red", linestyle="--")

# Label axes with units and a civil context title.
plt.xlabel("Time (s)")
plt.ylabel("Acceleration (g)")
plt.title("Bridge accelerometer signal with drift and impacts")

# Keep layout tidy for reports and notebooks.
plt.tight_layout()
plt.show()



## **2. Spatial Data Summaries**

### **2.1. Domain Conversion**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_B/image_02_01.jpg?v=1773550245" width="250">



>* Change measurement domain to reveal hidden patterns
>* Scale or frequency shows pavement roughness components

>* Convert spatial samples to frequency or wavelength.
>* Reveal repeating spacing, separate patterns from noise.

>* Convert domains to extract compact, comparable features
>* Separate trends, modes, and noise for decisions



In [ ]:
#@title Python Code - Domain Conversion

# Learn spatial domain conversion using simple synthetic profiles.
# Use NumPy FFT to reveal dominant spatial wavelengths.
# Summarize features for civil engineering surface roughness.

# pip install numpy pandas matplotlib seaborn.

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Set a deterministic seed for repeatable results.
np.random.seed(7)

# Define spatial sampling along a road centerline.
dx_m = 0.10
n_points = 512

# Build a distance axis with uniform spacing.
x_m = np.arange(n_points) * dx_m

# Create a profile with two repeating wavelengths.
amp_long = 8.0
amp_short = 3.0

wlen_long_m = 12.0
wlen_short_m = 2.5

# Add small measurement noise to mimic sensors.
noise_mm = np.random.normal(0.0, 0.8, n_points)

profile_mm = (
    amp_long * np.sin(2 * np.pi * x_m / wlen_long_m)
    + amp_short * np.sin(2 * np.pi * x_m / wlen_short_m)
    + noise_mm
)

# Store the spatial series in a small DataFrame.
df = pd.DataFrame({"x_m": x_m, "elev_mm": profile_mm})

# Remove the mean to focus on repeating structure.
y_mm = df["elev_mm"].to_numpy() - df["elev_mm"].mean()

# Compute the one sided FFT magnitude spectrum.
y_fft = np.fft.rfft(y_mm)
freq_cpm = np.fft.rfftfreq(n_points, d=dx_m)

mag = np.abs(y_fft) / n_points

# Convert frequency to wavelength, avoiding division by zero.
wavelength_m = np.full(freq_cpm.shape, np.inf)

valid = freq_cpm > 0
wavelength_m[valid] = 1.0 / freq_cpm[valid]

# Find the two strongest nonzero spectral peaks.
mag_nz = mag.copy()
mag_nz[0] = 0.0

peak1 = int(np.argmax(mag_nz))
mag_nz[peak1] = 0.0

peak2 = int(np.argmax(mag_nz))

# Compute simple spatial features for screening.
rms_mm = float(np.sqrt(np.mean(y_mm ** 2)))

band_short = (wavelength_m >= 1.0) & (wavelength_m <= 3.0)
band_long = (wavelength_m >= 8.0) & (wavelength_m <= 20.0)

energy = mag ** 2
short_ratio = float(energy[band_short].sum() / energy.sum())
long_ratio = float(energy[band_long].sum() / energy.sum())

# Print compact results without large arrays.
print("Samples:", n_points, "Spacing_m:", dx_m)
print("RMS_roughness_mm:", round(rms_mm, 2))

print("Peak1_wavelength_m:", round(float(wavelength_m[peak1]), 2))
print("Peak2_wavelength_m:", round(float(wavelength_m[peak2]), 2))

print("Energy_ratio_short_1to3m:", round(short_ratio, 3))
print("Energy_ratio_long_8to20m:", round(long_ratio, 3))

# Prepare a tidy spectrum table for plotting.
df_spec = pd.DataFrame(
    {"wavelength_m": wavelength_m, "magnitude": mag}
)

# Keep a practical wavelength range for visualization.
df_plot = df_spec[(df_spec["wavelength_m"] >= 0.5) & (df_spec["wavelength_m"] <= 30.0)]

# Plot magnitude versus wavelength to show dominant scales.
sns.set_theme(style="whitegrid")

plt.figure(figsize=(8, 4))
plt.plot(df_plot["wavelength_m"], df_plot["magnitude"], color="navy")

plt.xlabel("Wavelength (m)")
plt.ylabel("FFT magnitude")

plt.title("Domain conversion: spatial profile to wavelength spectrum")
plt.xlim(0.5, 30.0)

plt.tight_layout()
plt.show()



### **2.2. Spatial Data Reduction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_B/image_02_02.jpg?v=1773550280" width="250">



>* Shrink spatial measurements, keep important patterns.
>* Summarize points into segments, zones, or grids.

>* Group measurements by areas, then summarize statistics.
>* Standardizes comparisons, reduces uneven sampling effects.

>* Match summary scale to engineering decisions
>* Preserve key extremes; balance detail and usability



In [ ]:
#@title Python Code - Spatial Data Reduction

# Learn spatial reduction using simple grid aggregation.
# Summarize dense points into manageable engineering cells.
# Visualize reduced patterns with one clear heatmap.

# !pip -q install pandas numpy matplotlib seaborn

# Import core tools for arrays and tables.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fix randomness for repeatable spatial measurements.
rng = np.random.default_rng(7)

# Create synthetic corridor sensor points with noise.
n_points = 1200
x_m = rng.uniform(0, 1000, n_points)

# Create y coordinates and a spatially varying signal.
y_m = rng.uniform(0, 400, n_points)

# Build a smooth field plus random measurement noise.
base = 50 + 10 * np.sin(2 * np.pi * x_m / 500)

# Add a hotspot region to mimic localized distress.
hotspot = 25 * np.exp(-((x_m - 700) ** 2) / 20000)

# Combine effects into a measured condition index.
condition = base + hotspot + rng.normal(0, 4, n_points)

# Store points in a small Pandas dataframe.
df = pd.DataFrame({"x_m": x_m, "y_m": y_m, "condition": condition})

# Define grid cell size for spatial reduction.
cell_size_m = 50
x_bin = (df["x_m"] // cell_size_m).astype(int)

# Compute y bins using the same cell size.
y_bin = (df["y_m"] // cell_size_m).astype(int)

# Add grid indices for grouping and aggregation.
df["x_bin"] = x_bin

# Add y bin indices for spatial grouping.
df["y_bin"] = y_bin

# Aggregate points into cell summaries for decisions.
agg = df.groupby(["y_bin", "x_bin"]).agg(
    mean_condition=("condition", "mean"),
    std_condition=("condition", "std"),
    max_condition=("condition", "max"),
    n_points=("condition", "size"),
)

# Replace missing variability for single point cells.
agg["std_condition"] = agg["std_condition"].fillna(0.0)

# Report reduction ratio and basic summary counts.
n_cells = int(agg.shape[0])

# Print compact outputs without large tables.
print(f"Raw points: {n_points}, reduced cells: {n_cells}")

# Compute a simple reduction factor for storage intuition.
print(f"Reduction factor: {n_points / max(n_cells, 1):.1f}x")

# Show typical within cell variability for noise insight.
print(f"Median cell std: {agg['std_condition'].median():.2f}")

# Pivot mean values into a grid for plotting.
mean_grid = agg["mean_condition"].unstack("x_bin")

# Plot a heatmap of reduced mean condition values.
plt.figure(figsize=(9, 4))

# Use a robust colormap for engineering interpretation.
sns.heatmap(mean_grid, cmap="viridis", cbar_kws={"label": "Mean condition"})

# Label axes using grid cell indices.
plt.xlabel("Grid column index")

# Label y axis using grid row indices.
plt.ylabel("Grid row index")

# Add a clear title describing spatial reduction.
plt.title("Spatial data reduction: mean condition per grid cell")

# Render the single required plot.
plt.tight_layout(); plt.show()



### **2.3. Length Count Summaries**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_B/image_02_03.jpg?v=1773550313" width="250">



>* Summarize corridor conditions by total length.
>* Convert spaced readings into project-length totals.

>* Assign each reading its represented segment length
>* Bin categories, sum lengths for comparisons

>* Compare surveys, track change, prioritize fixes.
>* Ensure data quality; lengths guide decisions.



In [ ]:
#@title Python Code - Length Count Summaries

# Summarize corridor lengths using simple category bins.
# Handle irregular spacing to avoid biased totals.
# Visualize length totals for quick engineering decisions.

# Import core tools for tables and plotting.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set a deterministic seed for repeatable results.
rng = np.random.default_rng(7)

# Create irregular distance stations along a corridor.
steps = rng.uniform(5.0, 25.0, size=60)
distance_m = np.cumsum(steps)

# Simulate roughness values with a few severe zones.
roughness_iri = 1.2 + 0.02 * distance_m
roughness_iri = roughness_iri + rng.normal(0.0, 0.25, size=60)

# Add localized deterioration to mimic real corridors.
roughness_iri[15:22] = roughness_iri[15:22] + 1.2
roughness_iri[40:46] = roughness_iri[40:46] + 1.8

# Build a small dataframe for spatial measurements.
df = pd.DataFrame({"distance_m": distance_m, "IRI": roughness_iri})

# Compute segment lengths using midpoints between stations.
d_prev = df["distance_m"].shift(1)
d_next = df["distance_m"].shift(-1)

# Estimate each record's represented length defensively.
seg_len = (d_next - d_prev) / 2.0
seg_len.iloc[0] = df["distance_m"].iloc[1] - df["distance_m"].iloc[0]

# Fix the last segment length using last spacing.
seg_len.iloc[-1] = df["distance_m"].iloc[-1] - df["distance_m"].iloc[-2]
df["seg_len_m"] = seg_len.clip(lower=0.0)

# Define engineering bins for roughness condition categories.
bins = [0.0, 2.0, 3.0, np.inf]
labels = ["Acceptable", "Monitor", "Intervene"]

# Assign each segment to a condition category.
df["condition"] = pd.cut(df["IRI"], bins=bins, labels=labels)

# Aggregate total represented length by condition category.
length_by_cond = df.groupby("condition", observed=True)["seg_len_m"].sum()

# Compute corridor totals and percentage shares.
total_len = float(df["seg_len_m"].sum())
percent_by_cond = 100.0 * length_by_cond / total_len

# Print compact results without large tables.
print("Total corridor length (m):", round(total_len, 1))
print("Length by condition (m):")
print(length_by_cond.round(1).to_string())
print("Percent by condition (%):")
print(percent_by_cond.round(1).to_string())

# Prepare a tidy table for a simple bar plot.
plot_df = length_by_cond.reset_index()
plot_df.columns = ["condition", "length_m"]

# Make one clear plot showing length totals.
plt.figure(figsize=(6, 3.6))
sns.barplot(data=plot_df, x="condition", y="length_m", color="#4C72B0")

# Add labels and a readable title.
plt.ylabel("Total length (m)")
plt.xlabel("Condition category")
plt.title("Length count summary for roughness categories")

# Ensure the plot fits nicely in the notebook.
plt.tight_layout()
plt.show()



## **3. NumPy Array Statistics**

### **3.1. Column to Array**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_B/image_03_01.jpg?v=1773550352" width="250">



>* Convert a data column into NumPy array
>* Enables clean, fast vectorized measurement calculations

>* Check numeric values, missing data, and alignment.
>* Confirm array shape for single or multi-sensors.

>* Apply unit and sign changes consistently.
>* Clean signals fast with transparent array math.



In [ ]:
#@title Python Code - Column to Array

# Convert one Pandas column into a NumPy array.
# Clean missing values before vectorized engineering statistics.
# Confirm array shape and compute quick summary metrics.

# !pip -q install pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd

# Download a small civil engineering dataset file.
!wget -q -O geotech.csv "https://raw.githubusercontent.com/mhrafiei/data/main/geotech.csv"

# Load the table and preview only a few rows.
df = pd.read_csv("geotech.csv")
print("Columns:", list(df.columns)[:8])

# Select one measurement column and coerce to numeric.
series = pd.to_numeric(df["phi"], errors="coerce")
print("Nonmissing phi values:", int(series.notna().sum()))

# Drop missing values and convert to a NumPy array.
phi = series.dropna().to_numpy(dtype=float)
print("Array shape:", phi.shape, "dtype:", phi.dtype)

# Validate array size before computing statistics.
if phi.size < 5:
    raise ValueError("Too few phi samples for statistics")

# Compute vectorized statistics on the measurement array.
mean_phi = float(np.mean(phi))
std_phi = float(np.std(phi, ddof=1))

# Compute a simple transformation and a percentile.
phi_rad = np.deg2rad(phi)
p95_phi = float(np.percentile(phi, 95))

# Print compact results without dumping the full array.
print("Mean phi deg:", round(mean_phi, 2))
print("Std phi deg:", round(std_phi, 2))
print("95th phi deg:", round(p95_phi, 2))
print("Mean phi rad:", round(float(np.mean(phi_rad)), 4))



### **3.2. Vectorized Array Operations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_B/image_03_02.jpg?v=1773550377" width="250">



>* Apply one operation to all measurements.
>* Convert units, calibrate offsets, without loops.

>* Mask, flag, and fix bad values quickly
>* Combine aligned streams for derived calculations

>* Vectorize multidimensional transforms: detrend, normalize, gradients, grids.
>* Fast, readable workflows; iterate while preserving meaning.



In [ ]:
#@title Python Code - Vectorized Array Operations

# Learn vectorized operations on engineering measurement arrays.
# Apply masks, scaling, and derived signals without loops.
# Visualize results with one clear comparison plot.

# Uncomment installs only if your Colab lacks libraries.
# !pip -q install numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for reproducible synthetic sensor data.
rng = np.random.default_rng(7)
fs_hz = 50
n_samples = 500

# Create time vector and acceleration in milli-g units.
t_s = np.arange(n_samples) / fs_hz
acc_mg = 50 * np.sin(2 * np.pi * 1.2 * t_s)

# Add noise and a constant sensor offset.
acc_mg = acc_mg + rng.normal(0, 8, n_samples)
acc_mg = acc_mg + 12

# Convert units and remove offset using vectorized math.
g0 = 9.80665
acc_ms2 = (acc_mg / 1000) * g0
acc_ms2 = acc_ms2 - (12 / 1000) * g0

# Flag impossible spikes and replace them with NaN.
spike_mask = np.abs(acc_ms2) > 1.5
acc_clean = acc_ms2.copy()
acc_clean[spike_mask] = np.nan

# Compute vectorized statistics while ignoring missing values.
mean_val = np.nanmean(acc_clean)
std_val = np.nanstd(acc_clean)

# Standardize values and compute an energy-like feature.
z = (acc_clean - mean_val) / std_val
rms = np.sqrt(np.nanmean(acc_clean ** 2))

# Print a compact summary without printing large arrays.
print(f"Samples: {n_samples}, Sampling Hz: {fs_hz}")
print(f"Spikes flagged: {int(np.sum(spike_mask))}")
print(f"Mean m/s^2: {mean_val:.4f}, Std m/s^2: {std_val:.4f}")
print(f"RMS m/s^2: {rms:.4f}, Z-score range: {np.nanmin(z):.2f} to {np.nanmax(z):.2f}")

# Plot raw and cleaned signals to show masking effects.
plt.figure(figsize=(9, 3.5))
plt.plot(t_s, acc_ms2, label="Raw (offset removed)")
plt.plot(t_s, acc_clean, label="Cleaned (spikes -> NaN)")
plt.xlabel("Time (s)")
plt.ylabel("Acceleration (m/s^2)")
plt.title("Vectorized scaling, masking, and feature extraction")
plt.legend()
plt.tight_layout()
plt.show()



### **3.3. Descriptive Array Statistics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_02/Lecture_B/image_03_03.jpg?v=1773550412" width="250">



>* Summarize large sensor arrays quickly, consistently.
>* Mean and spread reveal baselines, variability, issues.

>* Compute stats along axes for sensors, time.
>* Handle missing data; relate spread to causes.

>* Use percentiles, medians, extremes for spikes.
>* Check skew, tails; create features, flag outliers.



In [ ]:
#@title Python Code - Descriptive Array Statistics

# Learn descriptive statistics for engineering sensor arrays.
# Use NumPy axis operations with missing values.
# Visualize distributions to support quick quality checks.

# Import core libraries for arrays and plotting.
import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic seed for reproducible example data.
rng = np.random.default_rng(7)

# Create time by sensor acceleration array in m/s^2.
acc = rng.normal(loc=0.0, scale=0.15, size=(600, 4))

# Add a transient event spike on one sensor channel.
acc[250:260, 2] = acc[250:260, 2] + 0.9

# Insert missing samples to mimic communication dropouts.
acc[100:110, 1] = np.nan

# Confirm array shape matches time samples and sensors.
print("Array shape (time, sensors):", acc.shape)

# Compute per sensor mean and standard deviation.
mean_s = np.nanmean(acc, axis=0)
std_s = np.nanstd(acc, axis=0)

# Compute robust per sensor median and percentiles.
median_s = np.nanmedian(acc, axis=0)
p95_s = np.nanpercentile(acc, 95, axis=0)

# Compute per sensor min and max ignoring missing values.
min_s = np.nanmin(acc, axis=0)
max_s = np.nanmax(acc, axis=0)

# Count missing values per sensor for data quality.
nan_count = np.isnan(acc).sum(axis=0)

# Print a compact summary table for four sensors.
print("Sensor  mean   std   median   p95   min   max   NaNs")

# Loop over sensors and print one line each.
for j in range(acc.shape[1]):
    print(
        "S" + str(j + 1),
        f"{mean_s[j]: .3f}",
        f"{std_s[j]: .3f}",
        f"{median_s[j]: .3f}",
        f"{p95_s[j]: .3f}",
        f"{min_s[j]: .3f}",
        f"{max_s[j]: .3f}",
        int(nan_count[j]),
    )

# Compute system wide response per time using RMS.
rms_time = np.sqrt(np.nanmean(acc * acc, axis=1))

# Plot histogram of RMS to show distribution and extremes.
plt.figure(figsize=(7, 4))

# Draw histogram with a reasonable number of bins.
plt.hist(rms_time, bins=30, color="steelblue", edgecolor="white")

# Add labels and a title for engineering interpretation.
plt.xlabel("RMS acceleration across sensors (m/s^2)")
plt.ylabel("Count")
plt.title("Descriptive statistics view: RMS distribution")

# Show the single required plot.
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Pandas & Arrays**</font>


In this lecture, you learned to:
- Load and visualize basic civil engineering data using Pandas, NumPy, matplotlib, and seaborn. 
- Perform simple numerical operations such as converting time-series from time domain to frequency domain or extracting time-series features. 
- Use NumPy arrays to compute vectorized statistics and transformations on engineering measurements. 

In the next Module (Module 3), we will go over 'Framing ML Problems'